In [1]:
!pip install --upgrade binance-historical-data plotly -q
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.9 MB/s eta 0:00:00


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from binance_historical_data import BinanceDataDumper
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
warnings.filterwarnings('ignore')

In [3]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data


In [4]:
data_frequency = "1d"
start_date = "2024-01-01"
end_date = "2025-07-01"
tickers =['BTCUSDT']




In [5]:
download_ticker(tickers[0], start_date, end_date, data_frequency)
crypto_data = process_csv_files(tickers[0], data_frequency)


---> Found overall tickers: 594
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20240101
---> End Date: 20250701


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/18 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/1 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 18 months 1 days
---> For BTCUSD new data saved for: 0 months 0 days


In [6]:
data = crypto_data[['open_time','Open','High','Low', 'Close','volume']].reset_index(drop=True)
data.columns = ['timestamp','open','high','low', 'close','volume']
data

,timestamp,open,high,low,close,volume
0,2024-01-02,44179.55,45879.63,44148.34,44946.91,65146.40661
1,2024-01-03,44946.91,45500.00,40750.00,42845.23,81194.55173
2,2024-01-04,42845.23,44729.58,42613.77,44151.10,48038.06334
3,2024-01-05,44151.10,44357.46,42450.00,44145.11,48075.25327
4,2024-01-06,44145.12,44214.42,43397.05,43968.32,17835.06144
...,...,...,...,...,...,...
524,2025-06-26,107340.59,108272.45,106562.50,106947.06,10573.27279
525,2025-06-27,106947.06,107735.34,106356.76,107047.59,12232.44042
526,2025-06-28,107047.58,107577.75,106811.51,107296.79,3282.17352
527,2025-06-29,107296.79,108528.50,107172.52,108356.93,6831.73640


In [7]:
import pandas as pd
import numpy as np

def backtest_strategy(data, a1_percent, a2_percent, trailing_sl_percent):
    """
    Backtest trading strategy based on previous candle color with percentage-based returns.

    Parameters:
    - data: DataFrame with OHLC data (columns: timestamp, open, high, low, close)
    - a1_percent: Entry offset (percentage of previous candle close)
    - a2_percent: Fixed stop loss (percentage of previous candle close)
    - trailing_sl_percent: Trailing stop loss (percentage of previous candle close)

    Returns:
    - trades: DataFrame of trades with entry, exit, side, and percentage return
    - summary: Dictionary with total trades, win rate, and total percentage return
    """
    # Initialize variables
    trades = []
    position = None
    entry_price = 0
    stop_loss = 0
    trailing_stop = 0

    # Ensure data is sorted by timestamp
    data = data.sort_values('timestamp').reset_index(drop=True)

    for i in range(1, len(data)):
        prev_candle = data.iloc[i-1]
        curr_candle = data.iloc[i]
        a1 = a1_percent * prev_candle['close']
        a2 = a2_percent * prev_candle['close']
        trailing_sl = trailing_sl_percent * prev_candle['close']

        # Determine if previous candle was green (close > open)
        is_green = prev_candle['close'] > prev_candle['open']

        # Entry logic
        if position is None:
            entry_level = (prev_candle['low'] + a1) if is_green else (prev_candle['high'] + a1)
            side = 'long' if is_green else 'short'

            # Check if current candle triggers entry
            if (side == 'long' and curr_candle['high'] >= entry_level) or \
               (side == 'short' and curr_candle['low'] <= entry_level):
                entry_price = entry_level
                position = side
                stop_loss = entry_price * (1 - a2_percent) if side == 'long' else entry_price * (1 + a2_percent)
                trailing_stop = entry_price * (1 - trailing_sl_percent) if side == 'long' else entry_price * (1 + trailing_sl_percent)

                trades.append({
                    'entry_time': curr_candle['timestamp'],
                    'entry_price': entry_price,
                    'side': side,
                    'stop_loss': stop_loss,
                    'trailing_stop': trailing_stop
                })

        # Exit logic
        elif position is not None:
            if position == 'long':
                # Update trailing stop
                trailing_stop = max(trailing_stop, curr_candle['close'] * (1 - trailing_sl_percent))

                # Check for stop loss or trailing stop hit
                if curr_candle['low'] <= stop_loss or curr_candle['low'] <= trailing_stop:
                    exit_price = min(stop_loss, trailing_stop, curr_candle['low'])
                    pct_return = (exit_price - entry_price) / entry_price
                    trades[-1].update({
                        'exit_time': curr_candle['timestamp'],
                        'exit_price': exit_price,
                        'pct_return': pct_return
                    })
                    position = None

            elif position == 'short':
                # Update trailing stop
                trailing_stop = min(trailing_stop, curr_candle['close'] * (1 + trailing_sl_percent))

                # Check for stop loss or trailing stop hit
                if curr_candle['high'] >= stop_loss or curr_candle['high'] >= trailing_stop:
                    exit_price = max(stop_loss, trailing_stop, curr_candle['high'])
                    pct_return = (entry_price - exit_price) / entry_price
                    trades[-1].update({
                        'exit_time': curr_candle['timestamp'],
                        'exit_price': exit_price,
                        'pct_return': pct_return
                    })
                    position = None

    # Convert trades to DataFrame for analysis
    trades_df = pd.DataFrame(trades)

    # Calculate summary statistics
    if not trades_df.empty:
        total_trades = len(trades_df)
        win_rate = len(trades_df[trades_df['pct_return'] > 0]) / total_trades
        total_pct_return = trades_df['pct_return'].sum()
    else:
        total_trades = 0
        win_rate = 0
        total_pct_return = 0

    return {
        'trades': trades_df,
        'summary': {
            'total_trades': total_trades,
            'win_rate': win_rate,
            'total_pct_return': total_pct_return
        }
    }

# # Example usage
# if __name__ == "__main__":
#     # Load sample OHLC data (replace with your data source)
#     # data = pd.read_csv('ohlc_data.csv')  # Expected columns: timestamp, open, high, low, close

#     # Parameters (as percentages)
#     a1_percent = 0.01  # 0.5% offset
#     a2_percent = 0.01  # 1% stop loss
#     trailing_sl_percent = 0.0075  # 0.75% trailing stop loss

#     # Run backtest
#     result = backtest_strategy(data, a1_percent, a2_percent, trailing_sl_percent)

#     # Print results
#     print("Backtest Summary:")
#     print(f"Total Trades: {result['summary']['total_trades']}")
#     print(f"Win Rate: {result['summary']['win_rate']:.2%}")
#     print(f"Total Percentage Return: {result['summary']['total_pct_return']:.2%}")
#     print("\nTrades:")
#     print(result['trades'])

In [12]:
import pandas as pd
import numpy as np
from itertools import product

def sweep_test_parameters(data, a1_range, a2_range, trailing_sl_range):
    """
    Perform a parameter sweep test for a1_percent, a2_percent, and trailing_sl_percent.

    Parameters:
    - data: DataFrame with OHLC data (columns: timestamp, open, high, low, close)
    - a1_range: List of a1_percent values to test (as decimals, e.g., 0.005 for 0.5%)
    - a2_range: List of a2_percent values to test (as decimals)
    - trailing_sl_range: List of trailing_sl_percent values to test (as decimals)

    Returns:
    - results_df: DataFrame with parameter combinations and their performance metrics
    """
    # Initialize results list
    results = []

    # Generate all combinations of parameters
    for a1_percent, a2_percent, trailing_sl_percent in product(a1_range, a2_range, trailing_sl_range):
        # Run backtest with current parameter combination
        result = backtest_strategy(data, a1_percent, a2_percent, trailing_sl_percent)

        # Store results
        results.append({
            'a1_percent': a1_percent,
            'a2_percent': a2_percent,
            'trailing_sl_percent': trailing_sl_percent,
            'total_trades': result['summary']['total_trades'],
            'win_rate': result['summary']['win_rate'],
            'total_pct_return': result['summary']['total_pct_return']
        })

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Sort by total percentage return (descending)
    results_df = results_df.sort_values('total_pct_return', ascending=False)

    return results_df

# Example usage
if __name__ == "__main__":
    # Load sample OHLC data (replace with your data source)
    # data = pd.read_csv('ohlc_data.csv')  # Expected columns: timestamp, open, high, low, close

    # Define parameter ranges (as percentages)
    a1_range = np.arange(0.1,1,0.1)  # 0.2%, 0.5%, 1%
    a2_range = np.arange(1,5,1)  # 0.5%, 1%, 1.5%
    trailing_sl_range = np.arange(0.1,1,0.1)  # 0.3%, 0.75%, 1.2%

    # Run sweep test
    results_df = sweep_test_parameters(data, a1_range, a2_range, trailing_sl_range)

    # Print results
    print("Parameter Sweep Test Results:")
    print(results_df)

    # Save results to CSV
    results_df.to_csv('sweep_test_results.csv', index=False)
    print("\nResults saved to 'sweep_test_results.csv'")

Parameter Sweep Test Results:
     a1_percent  a2_percent  trailing_sl_percent  total_trades  win_rate  \
296         0.9           1                  0.9             2       0.0   
44          0.2           1                  0.9             2       0.0   
260         0.8           1                  0.9             2       0.0   
80          0.3           1                  0.9             2       0.0   
3           0.1           1                  0.4             2       0.0   
..          ...         ...                  ...           ...       ...   
135         0.4           4                  0.1            29       0.0   
99          0.3           4                  0.1            29       0.0   
171         0.5           4                  0.1            29       0.0   
279         0.8           4                  0.1            29       0.0   
63          0.2           4                  0.1            29       0.0   

     total_pct_return  
296              -1.0  
44       